<a href="https://colab.research.google.com/github/stefdeg145/sma-rul-twin/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 0 — Regenerate the synthetic data in THIS notebook's session
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
C, m = 5.0, 2.0
strain_ranges = [0.010, 0.015, 0.020, 0.030, 0.045]
NOISE_LEVEL = 0.01
D_FAIL = 0.30

def cycles_to_failure(delta_eps_p):
    return C * (delta_eps_p ** (-m))

def simulate_condition(delta_eps_p, cond_id):
    Nf = int(cycles_to_failure(delta_eps_p))
    eps_rec_initial = 0.06
    cycles = np.arange(1, Nf + 1)
    fraction_life = cycles / Nf
    decay = D_FAIL * (fraction_life ** 1.5)
    eps_rec_clean = eps_rec_initial * (1.0 - decay)
    noise = rng.normal(0, NOISE_LEVEL * eps_rec_initial, size=Nf)
    eps_rec = eps_rec_clean + noise
    D = np.clip(1.0 - eps_rec / eps_rec[0], 0.0, 1.0)
    return pd.DataFrame({
        "condition_id": cond_id,
        "delta_eps_p": delta_eps_p,
        "Nf_true": Nf,
        "cycle": cycles,
        "eps_recoverable": eps_rec,
        "damage_D": D,
    })

frames = [simulate_condition(s, i) for i, s in enumerate(strain_ranges)]
df = pd.concat(frames, ignore_index=True)
df.to_csv("sma_synthetic.csv", index=False)
print(f"Wrote sma_synthetic.csv ({len(df):,} rows)")

Wrote sma_synthetic.csv (92,746 rows)


In [ ]:
# Cell 1 — Setup and load
import numpy as np
import pandas as pd

df = pd.read_csv("sma_synthetic.csv")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Conditions:", sorted(df.condition_id.unique()))
print("\nFirst few rows:")
print(df.head())
print("\nRows per condition:")
print(df.groupby("condition_id").size())

Shape: (92746, 6)
Columns: ['condition_id', 'delta_eps_p', 'Nf_true', 'cycle', 'eps_recoverable', 'damage_D']
Conditions: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

First few rows:
   condition_id  delta_eps_p  Nf_true  cycle  eps_recoverable  damage_D
0             0         0.01    50000      1         0.060183  0.000000
1             0         0.01    50000      2         0.059376  0.013406
2             0         0.01    50000      3         0.060450  0.000000
3             0         0.01    50000      4         0.060564  0.000000
4             0         0.01    50000      5         0.058829  0.022489

Rows per condition:
condition_id
0    50000
1    22222
2    12500
3     5555
4     2469
dtype: int64


In [ ]:
# Cell 2 — Feature engineering: smoothed damage + block-rate target ΔD
SMOOTH = 300   # smoothing window (cycles) — wide enough to beat the noise
BLOCK  = 200   # horizon (cycles) over which we measure the damage rise

df = df.sort_values(["condition_id", "cycle"]).reset_index(drop=True)

def engineer(group):
    group = group.sort_values("cycle").copy()
    group["D_smooth"] = (group["damage_D"]
                         .rolling(window=SMOOTH, center=True, min_periods=1)
                         .mean())
    group["delta_D_block"] = group["D_smooth"].shift(-BLOCK) - group["D_smooth"]
    return group

# Keep condition_id available inside the function AND after (no include_groups)
df = df.groupby("condition_id", group_keys=False)[df.columns.tolist()].apply(engineer)

df["life_frac"] = df["cycle"] / df["Nf_true"]

before = len(df)
df = df.dropna(subset=["delta_D_block"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows lacking a forward block target.")

neg = (df["delta_D_block"] < 0).mean()
print(f"Negative block-ΔD fraction: {neg:.4f}  (want low single digits)")
print(f"Block-ΔD range: [{df.delta_D_block.min():.2e}, {df.delta_D_block.max():.2e}]")
print("Block-ΔD mean by condition (should rise with condition_id):")
print(df.groupby("condition_id")["delta_D_block"].mean())
print("Rows per condition:")
print(df.groupby("condition_id").size())
print("\nColumns:", list(df.columns))
print(df[["condition_id","cycle","damage_D","D_smooth","delta_D_block","life_frac"]].head())

Dropped 1000 rows lacking a forward block target.
Negative block-ΔD fraction: 0.0443  (want low single digits)
Block-ΔD range: [-1.14e-03, 3.49e-02]
Block-ΔD mean by condition (should rise with condition_id):
condition_id
0    0.001188
1    0.002673
2    0.004710
3    0.010495
4    0.023873
Name: delta_D_block, dtype: float64
Rows per condition:
condition_id
0    49800
1    22022
2    12300
3     5355
4     2269
dtype: int64

Columns: ['condition_id', 'delta_eps_p', 'Nf_true', 'cycle', 'eps_recoverable', 'damage_D', 'D_smooth', 'delta_D_block', 'life_frac']
   condition_id  cycle  damage_D  D_smooth  delta_D_block  life_frac
0             0      1  0.000000  0.005435       0.000445    0.00002
1             0      2  0.013406  0.005430       0.000449    0.00004
2             0      3  0.000000  0.005482       0.000397    0.00006
3             0      4  0.000000  0.005576       0.000275    0.00008
4             0      5  0.022489  0.005591       0.000238    0.00010


In [ ]:
# Cell 3 — Sliding windows
WINDOW = 50   # timesteps of history the LSTM sees per example

FEATURES = ["damage_D", "D_smooth", "life_frac", "delta_eps_p"]
TARGET   = "delta_D_block"

def make_windows(group, window):
    g = group.sort_values("cycle")
    X = g[FEATURES].values
    y = g[TARGET].values
    cond = g["condition_id"].iloc[0]
    xs, ys, conds = [], [], []
    for i in range(len(g) - window):
        xs.append(X[i:i+window])          # window of history
        ys.append(y[i+window])            # target at the step after the window
        conds.append(cond)
    return np.array(xs), np.array(ys), np.array(conds)

Xs, ys, conds = [], [], []
for cid, group in df.groupby("condition_id"):
    xw, yw, cw = make_windows(group, WINDOW)
    Xs.append(xw); ys.append(yw); conds.append(cw)

X = np.concatenate(Xs)
y = np.concatenate(ys)
cond_arr = np.concatenate(conds)

print(f"X shape: {X.shape}   (examples, timesteps, features)")
print(f"y shape: {y.shape}")
print(f"Examples per condition: {np.bincount(cond_arr.astype(int))}")

X shape: (91496, 50, 4)   (examples, timesteps, features)
y shape: (91496,)
Examples per condition: [49750 21972 12250  5305  2219]


In [ ]:
# Cell 4 — Cycle-stratified split (NOT random — prevents leakage)
TEST_FRAC = 0.25   # last 25% of each condition's life -> test

train_idx, test_idx = [], []
for cid in np.unique(cond_arr):
    idx = np.where(cond_arr == cid)[0]      # windows are already in cycle order
    cut = int(len(idx) * (1 - TEST_FRAC))
    train_idx.extend(idx[:cut])             # early life -> train
    test_idx.extend(idx[cut:])              # late life  -> test

train_idx = np.array(train_idx); test_idx = np.array(test_idx)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {X_train.shape[0]} examples | Test: {X_test.shape[0]} examples")
print(f"Train target mean: {y_train.mean():.2e} | Test target mean: {y_test.mean():.2e}")

Train: 68620 examples | Test: 22876 examples
Train target mean: 2.72e-03 | Test target mean: 4.34e-03


In [ ]:
# Cell 5 — Normalize (fit on train only) and make tensors
import torch

feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std  = X_train.reshape(-1, X_train.shape[-1]).std(axis=0) + 1e-8

X_train_n = (X_train - feat_mean) / feat_std
X_test_n  = (X_test  - feat_mean) / feat_std

X_train_t = torch.tensor(X_train_n, dtype=torch.float32)
y_train_t = torch.tensor(y_train,   dtype=torch.float32).unsqueeze(-1)
X_test_t  = torch.tensor(X_test_n,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,    dtype=torch.float32).unsqueeze(-1)

print("Tensors ready:")
print(f"  X_train_t: {tuple(X_train_t.shape)}   y_train_t: {tuple(y_train_t.shape)}")
print(f"  X_test_t:  {tuple(X_test_t.shape)}    y_test_t:  {tuple(y_test_t.shape)}")
print(f"Feature means (pre-norm): {feat_mean}")

Tensors ready:
  X_train_t: (68620, 50, 4)   y_train_t: (68620, 1)
  X_test_t:  (22876, 50, 4)    y_test_t:  (22876, 1)
Feature means (pre-norm): [0.07950371 0.07951277 0.37136979 0.01454773]
